# LightGBM 변수 조합 선정 결과

## 1. 분석 목적

이상거래 탐지 모델에 어떤 변수 조합을 사용하는 것이 가장 적절한지 확인하기 위해
총 5개의 변수 조합을 비교하였다.

단순히 한 번의 학습/검증 결과만으로 최종 조합을 선택하지 않고,

1. 80:20 데이터 분할을 이용한 기본 성능 비교
2. 시간순 3-Fold 교차검증
3. LightGBM 하이퍼파라미터 튜닝
4. 각 변수 조합에 적합한 최적 임계값(Threshold) 탐색
5. 최종 성능 비교

순서로 검증하였다.

이 과정을 통해 특정 데이터 구간에서만 성능이 좋은 조합이 아니라,
시간이 지나 데이터의 특성이 달라져도 비교적 안정적으로 이상거래를 탐지할 수 있는
변수 조합을 선정하고자 하였다.


---

## 2. 평가 지표

이상거래 데이터는 정상거래에 비해 이상거래가 매우 적은 불균형 데이터이므로,
단순 정확도(Accuracy)보다는 다음 지표들을 중심으로 성능을 비교하였다.

### PR-AUC

Precision과 Recall의 관계를 종합적으로 평가하는 지표이다.

값이 높을수록 정상거래가 훨씬 많은 상황에서도
이상거래를 효과적으로 구분하고 있다고 볼 수 있다.

본 프로젝트에서는 주요 성능 평가 지표로 사용하였다.


### Precision

모델이 **"이상거래"라고 예측한 거래 중 실제 이상거래의 비율**이다.

Precision이 높다는 것은 정상거래를 이상거래라고 잘못 판단하는
오탐(False Positive)이 상대적으로 적다는 의미이다.


### Recall

**실제 이상거래 중 모델이 이상거래라고 찾아낸 비율**이다.

Recall이 높을수록 실제 이상거래를 놓치는 경우(False Negative)가 적다.

임계값은 모델이 출력한 이상거래 예측확률을 기준으로
정상거래와 이상거래를 구분하는 기준값이다.

예를 들어 임계값이 0.9라면 이상거래 예측확률이 0.9 이상인 거래를
이상거래로 분류한다.

- **임계값을 낮추면** 더 많은 거래를 이상거래로 분류하므로
  실제 이상거래를 찾아낼 가능성이 높아지고 일반적으로 Recall이 증가한다.
  대신 정상거래까지 이상거래로 판단하는 False Positive(FP)가 증가할 수 있다.

- **임계값을 높이면** 이상거래로 분류하는 기준이 엄격해지므로
  실제 이상거래를 놓치는 False Negative(FN)가 증가할 수 있고,
  일반적으로 Recall은 감소한다.

따라서 이상거래 탐지처럼 실제 이상거래를 놓치는 비용이 큰 문제에서는
Recall과 임계값의 관계를 함께 고려하는 것이 중요하다.


### F1-score

Precision과 Recall을 동시에 고려하는 지표이다.

따라서 이상거래를 많이 찾아내면서도
정상거래를 이상거래로 잘못 판단하는 문제를 함께 고려할 수 있다.

F1-score 역시 **임계값(Threshold)에 따라 달라진다.**

임계값을 낮추면 일반적으로 Recall은 높아지지만,
정상거래까지 이상거래로 예측하는 경우가 증가하면서
Precision이 낮아질 수 있다.

반대로 임계값을 높이면 이상거래 판정 기준이 엄격해져
Precision이 높아질 수 있지만,
실제 이상거래를 놓치면서 Recall이 낮아질 수 있다.

즉, 임계값을 무조건 낮추거나 높인다고 해서
F1-score가 계속 좋아지는 것은 아니다.

F1-score는 Precision과 Recall의 균형을 평가하기 때문에,
여러 임계값을 적용했을 때 **Precision과 Recall의 균형이 가장 좋은 지점에서
F1-score가 최대가 되는 임계값**을 찾을 수 있다.


---

# 3. 1차 비교: 80:20 성능

먼저 동일한 조건에서 5개의 변수 조합을 비교하였다.

| 변수 조합 | PR-AUC | Precision | Recall | F1-score | FP | FN |
|---|---:|---:|---:|---:|---:|---:|
| 조합 1 | 0.972656 | 0.899685 | 0.950033 | 0.924174 | 159 | 75 |
| 조합 2 | 0.971890 | 0.884758 | 0.951366 | 0.916854 | 186 | 73 |
| 조합 3 | 0.971038 | 0.878844 | 0.952032 | 0.913975 | 197 | 72 |
| 조합 4 | 0.965424 | 0.806215 | 0.950700 | 0.872516 | 343 | 74 |
| 조합 5 | **0.982701** | **0.968793** | 0.930713 | **0.949371** | **45** | 104 |

80:20 결과만 보면 **조합 5의 성능이 가장 우수하였다.**

특히 PR-AUC와 F1-score가 가장 높았으며,
False Positive(FP) 역시 45건으로 가장 적었다.

따라서 단일 80:20 분할만 평가했다면 조합 5가 가장 좋은 선택으로 보일 수 있다.

하지만 한 번의 데이터 분할에서 높은 성능을 기록했다고 해서
새로운 시점의 데이터에서도 동일하게 좋은 성능을 보인다고 단정할 수 없다.

따라서 시간 변화에 따른 모델의 안정성을 확인하기 위해
추가적으로 **시간순 3-Fold 교차검증**을 실시하였다.


---

# 4. 시간순 3-Fold 교차검증 및 하이퍼파라미터 튜닝

시간순 교차검증은 과거 데이터를 이용하여 모델을 학습하고
그보다 미래의 데이터를 이용하여 검증하는 방식이다.

일반적인 랜덤 분할과 달리 시간의 순서를 유지하기 때문에,
실제 서비스 환경에서 **과거 거래를 학습한 모델이 미래 거래를 얼마나 잘 예측하는지**
확인하는 데 적합하다.

LightGBM의 하이퍼파라미터를 조정한 후 주요 후보 조합을 비교한 결과는 다음과 같다.

| 변수 조합 | 최종 튜닝 설정 | Mean PR-AUC | Std PR-AUC |
|---|---|---:|---:|
| **조합 3** | num_leaves=15, min_child_samples=100 | **0.974956** | 0.004053 |
| 조합 1 | num_leaves=15, min_child_samples=100 | 0.974500 | **0.003208** |
| 조합 5 | num_leaves=63, min_child_samples=300 | 0.965676 | 0.006482 |

### Mean PR-AUC

3개의 Fold에서 얻은 PR-AUC의 평균이다.

평균이 높을수록 여러 시간 구간에서 전반적으로 높은 성능을 보였다는 의미이다.

**조합 3이 0.974956으로 가장 높은 Mean PR-AUC를 기록하였다.**


### Std PR-AUC

3개의 Fold에서 나타난 PR-AUC의 표준편차이다.

값이 작을수록 시간 구간이 달라져도 성능 변화가 크지 않다는 의미이다.

조합 1의 표준편차가 가장 작았지만,
조합 3 역시 0.004053으로 낮은 수준을 유지하면서
가장 높은 평균 PR-AUC를 기록하였다.


---

# 5. 최적 임계값 탐색

모델은 각 거래가 이상거래일 **확률**을 계산한다.

예를 들어 모델이 어떤 거래의 이상거래 확률을 0.85라고 예측했다고 하더라도,
어느 확률부터 실제 이상거래로 판단할지는 별도로 결정해야 한다.

이 기준을 **임계값(Threshold)**이라고 한다.

모든 변수 조합에 동일한 임계값을 강제로 적용하지 않고,
각 조합의 Validation 예측확률과 실제값을 통합한 뒤
**F1-score가 가장 높아지는 임계값을 조합별로 탐색하였다.**

최종 결과는 다음과 같다.

| 순위 | 변수 조합 | 최적 Threshold | PR-AUC | Precision | Recall | F1-score | FP | FN |
|---:|---|---:|---:|---:|---:|---:|---:|---:|
| **1** | **조합 3** | **0.971298** | **0.975170** | **0.961375** | 0.920087 | **0.940278** | **136** | 294 |
| 2 | 조합 1 | 0.948156 | 0.973112 | 0.945781 | **0.929329** | 0.937483 | 196 | **260** |
| 3 | 조합 5 | 0.817684 | 0.959111 | 0.955768 | 0.898614 | 0.926310 | - | - |


---

# 6. 최종 선정: 조합 3

## 왜 조합 3을 선택했는가?

최종적으로 **조합 3을 최종 변수 조합으로 선정하였다.**

가장 중요한 이유는 단순히 특정 데이터 분할에서 높은 성능을 기록한 것이 아니라,
시간순 교차검증과 하이퍼파라미터 튜닝을 거친 이후에도
우수한 종합 성능을 보였기 때문이다.

조합 3의 최종 성능은 다음과 같다.

- **PR-AUC: 0.975170 → 후보 중 1위**
- **Precision: 0.961375 → 후보 중 1위**
- **Recall: 0.920087**
- **F1-score: 0.940278 → 후보 중 1위**
- **False Positive: 136건**

특히 Precision과 Recall 중 어느 한쪽에만 치우치지 않고,
두 지표의 균형을 나타내는 F1-score에서 가장 높은 성능을 기록하였다.

따라서 조합 3은 **이상거래를 충분히 탐지하면서 정상거래를 이상거래로 잘못 판단하는 문제도 비교적 효과적으로 통제한 조합**으로 판단하였다.


---

# 7. 다른 조합을 선택하지 않은 이유

## 조합 1

조합 1은 매우 우수한 후보였다.

특히 Recall이 **0.929329**로 조합 3보다 높았으며,
3-Fold PR-AUC의 표준편차도 **0.003208**로 가장 작았다.

즉, 실제 이상거래를 찾아내는 능력과 시간에 따른 안정성 측면에서는 강점이 있었다.

하지만 최종 비교에서는 다음과 같은 차이가 나타났다.

- 조합 1 F1-score: **0.937483**
- 조합 3 F1-score: **0.940278**

또한 False Positive는

- 조합 1: **196건**
- 조합 3: **136건**

으로 조합 1이 60건 더 많았다.

즉 조합 1은 실제 이상거래를 조금 더 많이 찾아냈지만,
그 과정에서 정상거래를 이상거래로 잘못 판단하는 경우도 더 많았다.

따라서 **Precision과 Recall의 전체적인 균형에서는 조합 3이 더 우수하다고 판단하였다.**


---

## 조합 2

80:20 기본 성능 비교에서 조합 2는 다음과 같은 결과를 기록하였다.

- PR-AUC: 0.971890
- Precision: 0.884758
- Recall: 0.951366
- F1-score: 0.916854
- FP: 186건

조합 2는 조합 1에 `merchant_change_count` 변수를 추가한 형태였지만,
변수를 추가했음에도 조합 1보다 F1-score가 낮았다.

즉 **변수를 추가한 만큼의 성능 향상이 확인되지 않았다.**

따라서 더 복잡한 변수 구성을 사용할 실익이 크지 않다고 판단하여
후속 주요 후보에서 제외하였다.


---

## 조합 4

조합 4는 변수 수를 크게 줄인 비교적 단순한 조합이다.

하지만 80:20 비교에서

- PR-AUC: **0.965424**
- Precision: **0.806215**
- F1-score: **0.872516**
- FP: **343건**

으로 5개 조합 중 가장 낮은 수준의 성능을 보였다.

특히 FP가 343건으로 크게 증가하였다.

이는 정상거래를 이상거래로 잘못 판단하는 경우가 다른 조합보다 많았다는 의미이다.

따라서 변수 수를 줄여 모델을 단순하게 만드는 장점보다
**중요한 정보가 제외되면서 발생한 성능 저하가 더 크다고 판단하여 탈락시켰다.**


---

## 조합 5

조합 5는 가장 주의 깊게 비교해야 하는 조합이다.

80:20 기본 비교에서는 오히려 조합 5가 가장 좋은 성능을 보였다.

- PR-AUC: **0.982701**
- Precision: **0.968793**
- F1-score: **0.949371**
- FP: **45건**

따라서 80:20 결과만 본다면 조합 5를 선택하는 것이 자연스럽다.

하지만 시간순 3-Fold 교차검증과 튜닝을 실시한 결과 상황이 달라졌다.

조합 5의 Mean PR-AUC는 **0.965676**으로 조합 3의 **0.974956**보다 낮아졌으며,
Fold별 PR-AUC 표준편차도 **0.006482**로 주요 후보 중 가장 컸다.

이는 특정 80:20 구간에서는 매우 좋은 성능을 보였지만,
**시간 구간이 달라질 때 성능이 상대적으로 덜 안정적이었다는 것을 의미한다.**

최적 임계값을 적용한 최종 결과에서도

- PR-AUC: 0.959111
- Recall: 0.898614
- F1-score: 0.926310

으로 조합 3보다 낮았다.

따라서 단일 데이터 분할에서의 높은 성능보다는
**시간 변화에도 비교적 안정적으로 유지되는 성능을 더 중요하게 평가하여 조합 5를 최종 선택하지 않았다.**


---

# 8. 최종 결론

초기 80:20 비교에서는 조합 5가 가장 높은 성능을 기록하였다.

그러나 한 번의 데이터 분할 결과만으로 최종 변수 조합을 결정하면
특정 시기의 데이터 특성에 지나치게 의존할 가능성이 있다.

따라서 시간순 3-Fold 교차검증과 LightGBM 하이퍼파라미터 튜닝을 추가로 실시하고,
각 조합의 Validation 예측확률을 통합하여 최적 임계값까지 탐색하였다.

그 결과 **조합 3이 최종 PR-AUC, Precision, F1-score에서 가장 우수한 성능을 기록하였다.**

또한 조합 1보다 False Positive가 적었으며,
조합 5보다 시간순 교차검증에서 높은 평균 PR-AUC와 낮은 성능 변동성을 보였다.

따라서 본 프로젝트에서는 **단일 시점에서의 최고 성능보다 시간 변화에 대한 일반화 성능과 Precision-Recall 간 균형을 우선적으로 고려하여 조합 3을 최종 변수 조합으로 선정하였다.**

> **최종 선정 변수 조합: 조합 3**